# 00. LST 후보 기준 생성

기존 `air_lst_is_candidate_true.xlsx`는 대기질 결합용 후보였으므로 최종 LST-only 실험에서는 쓰지 않는다.

이 노트북은 새 기준으로 `attempt2/lst_candidates.csv`를 만든다.

기준:
- 도시/관심지역 중심점마다 60m x 256 x 256 ROI를 만든다.
- Landsat 8/9 L2 장면을 날짜 범위에서 검색한다.
- ROI 안의 구름 비율이 80% 이상인 장면은 LST정보가 무효하므로 제거한다. (최대한도) 즉 `clear_fraction_roi >= 0.20`인 장면만 사용한다.
- 같은 날짜 11:00 KST 기준 AWS temp/wind/rain/humidity 가용 지점 밀도가 하한 이상인 장면만 사용한다.
- 모든 ROI를 끝까지 수집한 뒤 `ROI-year-season`마다 1개씩 뽑는다. 계절 안에서는 ROI별 목표 날짜를 분산시키고, 이미 쓰인 날짜에는 penalty를 줘 날짜 중복을 줄인다.
- 같은 city-date에 여러 Landsat path/row가 잡히면 clear fraction이 가장 높은 scene 하나만 남긴다.
- 날짜 기준으로 train/val/test split을 부여한다.
- 출력 컬럼은 통합 노트북이 바로 읽을 수 있는 schema로 저장한다.


## 후보 다양화 및 민통선 제외 정책

- ROI당 제한을 `year max 4`, `year-season max 2`로 낮춰 같은 중심점 반복을 줄인다.
- 중심점은 해안, 내륙 대도시, 강/호수, 분지, 산지 영향권을 섞는다.
- EGIS 중분류/세분류 WMS가 공백으로 들어오는 민통선 북부는 후보 수집 전에 제외한다.
- 서부권 제외 기준은 `고양-의정부-동두천-포천` cutoff line, 동부권은 `정암해수욕장-낙산해수욕장 중간` cutoff point다.


In [11]:
# 필요하면 Colab에서 먼저 실행
%pip install -q earthengine-api requests pandas numpy tqdm openpyxl pyproj

from __future__ import annotations

from pathlib import Path
import hashlib
import math
import os
import re
import time
from getpass import getpass

import ee
import numpy as np
import pandas as pd
import requests
from pyproj import Transformer
from tqdm.auto import tqdm


Note: you may need to restart the kernel to use updated packages.


## 1. 설정

In [ ]:
PROJECT_DIR = Path.cwd()
if PROJECT_DIR.name in {"final", "attempt1", "attempt2", "attempt3"}:
    PROJECT_DIR = PROJECT_DIR.parent
OUT_CSV = PROJECT_DIR / "attempt3" / "lst_candidates.csv"
OUT_XLSX = PROJECT_DIR / "attempt3" / "lst_candidates.xlsx"

GRID_N = 256
PIXEL_SIZE_M = 60
CRS_BASE = "EPSG:32653"

START_DATE = "2016-01-01"
END_DATE = "2025-12-31"
MAX_SCENE_CLOUD_COVER = 80
MAX_ROI_CLOUD_FRACTION = 0.80
MIN_CLEAR_FRACTION = 1.0 - MAX_ROI_CLOUD_FRACTION
# 기본 샘플링 단위: ROI-year-season마다 1개. attempt3 목표는 약 100 ROI x 10년 x 4계절 = 최대 약 4000개.
# AWS 밀도 검사는 장면별로 하지 않고, ROI별 고정 기준일 1회만 수행한다.
# ROI가 2016-01-01 11:00 AWS 밀도 기준을 통과하면 해당 ROI의 EE 대표 후보를 모두 살린다.
SAMPLES_PER_CITY_YEAR_SEASON = 1
PRE_AWS_SCENES_PER_ROI_YEAR_SEASON = 1
AWS_DENSITY_REFERENCE_DATE = "2016-01-01"
DATE_OVERLAP_PENALTY = 1000.0
ROI_BALANCE_SEED = 20260614
HOUR_KST = "1100"
TARGET_ROI_COUNT = 100
MIN_ROI_GAP_M = 0.0


# 기후데이터 후보 하한. ROI 주변 50km 안에서 각 변수군별 최소 5개 AWS가 있고,
# 가장 가까운 AWS가 30km 안에 있으면 IDW 기반 기후장이 과도하게 외삽되지 않는다고 본다.
AWS_DENSITY_RADIUS_KM = 50
MIN_AWS_STATIONS_PER_GROUP = 5
MAX_NEAREST_AWS_DISTANCE_KM = 30

LANDSAT_COLLECTIONS = [
    "LANDSAT/LC08/C02/T1_L2",
    "LANDSAT/LC09/C02/T1_L2",
]

# 새 후보 기준의 출발점이다. AWS 측정소가 비교적 있고, 수도권/내륙/해안/산지 영향이 섞이도록 잡았다.
# latitude/longitude는 패치 중심점이다.
CITY_CENTERS = [
    # 수도권/내륙 대도시
    {"city": "seoul", "latitude": 37.5665, "longitude": 126.9780, "roi_group": "metro_inland", "water_context": "river", "terrain_context": "plain"},
    {"city": "incheon", "latitude": 37.4563, "longitude": 126.7052, "roi_group": "coastal_metro", "water_context": "coast", "terrain_context": "plain"},
    {"city": "suwon", "latitude": 37.2636, "longitude": 127.0286, "roi_group": "metro_inland", "water_context": "river", "terrain_context": "plain"},
    {"city": "daejeon", "latitude": 36.3504, "longitude": 127.3845, "roi_group": "metro_inland", "water_context": "river", "terrain_context": "basin"},
    {"city": "daegu", "latitude": 35.8714, "longitude": 128.6014, "roi_group": "basin_city", "water_context": "river", "terrain_context": "basin"},
    {"city": "gwangju", "latitude": 35.1595, "longitude": 126.8526, "roi_group": "metro_inland", "water_context": "river", "terrain_context": "plain"},

    # 호수/강/분지/산지 영향
    {"city": "chuncheon", "latitude": 37.8813, "longitude": 127.7298, "roi_group": "lake_basin", "water_context": "lake", "terrain_context": "basin"},
    {"city": "wonju", "latitude": 37.3422, "longitude": 127.9202, "roi_group": "mountain_inland", "water_context": "river", "terrain_context": "mountain_basin"},
    {"city": "jecheon", "latitude": 37.1326, "longitude": 128.1910, "roi_group": "lake_mountain", "water_context": "lake", "terrain_context": "mountain"},
    {"city": "chungju", "latitude": 36.9910, "longitude": 127.9259, "roi_group": "lake_basin", "water_context": "lake", "terrain_context": "basin"},
    {"city": "cheongju", "latitude": 36.6424, "longitude": 127.4890, "roi_group": "inland_plain", "water_context": "river", "terrain_context": "plain"},
    {"city": "sejong", "latitude": 36.4800, "longitude": 127.2890, "roi_group": "river_inland", "water_context": "river", "terrain_context": "plain"},
    {"city": "jeonju", "latitude": 35.8242, "longitude": 127.1480, "roi_group": "inland_plain", "water_context": "river", "terrain_context": "plain"},
    {"city": "namwon", "latitude": 35.4164, "longitude": 127.3904, "roi_group": "mountain_basin", "water_context": "river", "terrain_context": "basin"},
    {"city": "andong", "latitude": 36.5684, "longitude": 128.7294, "roi_group": "river_basin", "water_context": "river_lake", "terrain_context": "basin"},
    {"city": "jinju", "latitude": 35.1800, "longitude": 128.1076, "roi_group": "river_basin", "water_context": "river", "terrain_context": "basin"},

    # 동해/남해/서해 해안권
    {"city": "gangneung", "latitude": 37.7519, "longitude": 128.8761, "roi_group": "east_coast", "water_context": "coast", "terrain_context": "coastal_mountain"},
    {"city": "donghae", "latitude": 37.5247, "longitude": 129.1143, "roi_group": "east_coast", "water_context": "coast", "terrain_context": "coastal_mountain"},
    {"city": "samcheok", "latitude": 37.4499, "longitude": 129.1652, "roi_group": "east_coast", "water_context": "coast", "terrain_context": "coastal_mountain"},
    {"city": "pohang", "latitude": 36.0190, "longitude": 129.3435, "roi_group": "east_coast_industrial", "water_context": "coast", "terrain_context": "coastal_plain"},
    {"city": "ulsan", "latitude": 35.5384, "longitude": 129.3114, "roi_group": "coastal_industrial", "water_context": "coast", "terrain_context": "coastal_plain"},
    {"city": "busan", "latitude": 35.1796, "longitude": 129.0756, "roi_group": "coastal_metro", "water_context": "coast", "terrain_context": "coastal_mountain"},
    {"city": "changwon", "latitude": 35.2279, "longitude": 128.6811, "roi_group": "south_coast_industrial", "water_context": "coast", "terrain_context": "coastal_plain"},
    {"city": "gimhae", "latitude": 35.2285, "longitude": 128.8894, "roi_group": "river_coastal", "water_context": "river_coast", "terrain_context": "plain"},
    {"city": "gunsan", "latitude": 35.9677, "longitude": 126.7366, "roi_group": "west_coast", "water_context": "coast_river", "terrain_context": "coastal_plain"},
    {"city": "mokpo", "latitude": 34.8118, "longitude": 126.3922, "roi_group": "southwest_coast", "water_context": "coast", "terrain_context": "coastal_plain"},
    {"city": "yeosu", "latitude": 34.7604, "longitude": 127.6622, "roi_group": "south_coast", "water_context": "coast", "terrain_context": "coastal_hill"},
    {"city": "suncheon", "latitude": 34.9506, "longitude": 127.4875, "roi_group": "bay_wetland", "water_context": "bay_wetland", "terrain_context": "coastal_plain"},

    # attempt2 확장 ROI: 해안/내륙/강·호수/산지·분지 표본을 더 넣어 지형-수역 조합을 늘린다.
    {"city": "yongin", "latitude": 37.2411, "longitude": 127.1776, "roi_group": "metro_inland", "water_context": "river", "terrain_context": "plain"},
    {"city": "ansan", "latitude": 37.3219, "longitude": 126.8309, "roi_group": "west_coast_industrial", "water_context": "coast", "terrain_context": "coastal_plain"},
    {"city": "pyeongtaek", "latitude": 36.9921, "longitude": 127.1127, "roi_group": "west_coast_industrial", "water_context": "coast", "terrain_context": "coastal_plain"},
    {"city": "icheon", "latitude": 37.2723, "longitude": 127.4350, "roi_group": "inland_plain", "water_context": "river", "terrain_context": "plain"},
    {"city": "cheonan", "latitude": 36.8151, "longitude": 127.1139, "roi_group": "inland_plain", "water_context": "river", "terrain_context": "plain"},
    {"city": "asan", "latitude": 36.7898, "longitude": 127.0018, "roi_group": "river_inland", "water_context": "river_lake", "terrain_context": "plain"},
    {"city": "dangjin", "latitude": 36.8931, "longitude": 126.6283, "roi_group": "west_coast_industrial", "water_context": "coast", "terrain_context": "coastal_plain"},
    {"city": "seosan", "latitude": 36.7848, "longitude": 126.4503, "roi_group": "west_coast", "water_context": "coast", "terrain_context": "coastal_plain"},
    {"city": "boryeong", "latitude": 36.3334, "longitude": 126.6128, "roi_group": "west_coast", "water_context": "coast", "terrain_context": "coastal_plain"},
    {"city": "iksan", "latitude": 35.9483, "longitude": 126.9576, "roi_group": "inland_plain", "water_context": "river", "terrain_context": "plain"},
    {"city": "naju", "latitude": 35.0161, "longitude": 126.7108, "roi_group": "river_plain", "water_context": "river", "terrain_context": "plain"},
    {"city": "damyang", "latitude": 35.3211, "longitude": 126.9882, "roi_group": "inland_plain", "water_context": "river", "terrain_context": "plain"},
    {"city": "gumi", "latitude": 36.1195, "longitude": 128.3446, "roi_group": "river_industrial", "water_context": "river", "terrain_context": "basin"},
    {"city": "gimcheon", "latitude": 36.1398, "longitude": 128.1136, "roi_group": "river_basin", "water_context": "river", "terrain_context": "basin"},
    {"city": "sangju", "latitude": 36.4109, "longitude": 128.1591, "roi_group": "river_inland", "water_context": "river", "terrain_context": "plain"},
    {"city": "gyeongsan", "latitude": 35.8251, "longitude": 128.7415, "roi_group": "basin_city", "water_context": "river", "terrain_context": "basin"},
    {"city": "gyeongju", "latitude": 35.8562, "longitude": 129.2247, "roi_group": "east_coast_basin", "water_context": "coast_river", "terrain_context": "coastal_basin"},
    {"city": "danyang", "latitude": 36.9845, "longitude": 128.3655, "roi_group": "lake_mountain", "water_context": "river_lake", "terrain_context": "mountain"},
    {"city": "yeongwol", "latitude": 37.1838, "longitude": 128.4617, "roi_group": "mountain_river", "water_context": "river", "terrain_context": "mountain"},
    {"city": "taebaek", "latitude": 37.1641, "longitude": 128.9856, "roi_group": "mountain_inland", "water_context": "river", "terrain_context": "mountain"},
    {"city": "pyeongchang", "latitude": 37.3705, "longitude": 128.3904, "roi_group": "mountain_inland", "water_context": "river", "terrain_context": "mountain"},
    {"city": "miryang", "latitude": 35.5038, "longitude": 128.7465, "roi_group": "river_basin", "water_context": "river", "terrain_context": "basin"},
    {"city": "geoje", "latitude": 34.8806, "longitude": 128.6213, "roi_group": "island_coast", "water_context": "coast", "terrain_context": "coastal_hill"},
    {"city": "tongyeong", "latitude": 34.8544, "longitude": 128.4330, "roi_group": "south_coast", "water_context": "coast", "terrain_context": "coastal_hill"},
    {"city": "sacheon", "latitude": 35.0038, "longitude": 128.0642, "roi_group": "south_coast_industrial", "water_context": "coast", "terrain_context": "coastal_plain"},
    {"city": "hadong", "latitude": 35.0672, "longitude": 127.7513, "roi_group": "river_coast", "water_context": "river_coast", "terrain_context": "coastal_hill"},
    {"city": "boseong", "latitude": 34.7715, "longitude": 127.0801, "roi_group": "south_coast", "water_context": "coast", "terrain_context": "coastal_hill"},
    {"city": "haenam", "latitude": 34.5735, "longitude": 126.5989, "roi_group": "southwest_coast", "water_context": "coast", "terrain_context": "coastal_plain"},
]

# attempt3: 같은 도시 안에서도 서로 겹치지 않는 ROI를 추가해 수역/해안/분지/산지/도심 다양성을 늘린다.
# `city`는 ROI 식별자라서 unique해야 하고, 실제 도시명은 `base_city`로 보존한다.
EXTRA_ROI_CENTERS = [
    {"city": "seoul_han_river", "base_city": "seoul", "latitude": 37.5250, "longitude": 126.9350, "roi_group": "metro_river", "water_context": "river", "terrain_context": "plain"},
    {"city": "seoul_northeast", "base_city": "seoul", "latitude": 37.6550, "longitude": 127.0750, "roi_group": "metro_mountain_edge", "water_context": "river", "terrain_context": "mountain_edge"},
    {"city": "seoul_southeast", "base_city": "seoul", "latitude": 37.4850, "longitude": 127.0850, "roi_group": "metro_inland", "water_context": "river", "terrain_context": "plain"},
    {"city": "incheon_coast", "base_city": "incheon", "latitude": 37.4050, "longitude": 126.6000, "roi_group": "coastal_metro", "water_context": "coast", "terrain_context": "coastal_plain"},
    {"city": "incheon_airport", "base_city": "incheon", "latitude": 37.4600, "longitude": 126.4400, "roi_group": "island_coast", "water_context": "coast", "terrain_context": "coastal_plain"},
    {"city": "suwon_west", "base_city": "suwon", "latitude": 37.2500, "longitude": 126.9250, "roi_group": "metro_inland", "water_context": "river", "terrain_context": "plain"},
    {"city": "daejeon_west", "base_city": "daejeon", "latitude": 36.3550, "longitude": 127.2550, "roi_group": "metro_basin", "water_context": "river", "terrain_context": "basin"},
    {"city": "daejeon_south", "base_city": "daejeon", "latitude": 36.2450, "longitude": 127.3850, "roi_group": "metro_basin", "water_context": "river", "terrain_context": "basin_mountain_edge"},
    {"city": "daegu_west", "base_city": "daegu", "latitude": 35.8700, "longitude": 128.4800, "roi_group": "basin_city", "water_context": "river", "terrain_context": "basin"},
    {"city": "daegu_east", "base_city": "daegu", "latitude": 35.9000, "longitude": 128.7150, "roi_group": "basin_mountain_edge", "water_context": "river", "terrain_context": "mountain_edge"},
    {"city": "gwangju_west", "base_city": "gwangju", "latitude": 35.1650, "longitude": 126.7300, "roi_group": "metro_plain", "water_context": "river", "terrain_context": "plain"},
    {"city": "gwangju_south", "base_city": "gwangju", "latitude": 35.0550, "longitude": 126.8550, "roi_group": "metro_plain", "water_context": "river", "terrain_context": "plain"},
    {"city": "chuncheon_lake", "base_city": "chuncheon", "latitude": 37.8300, "longitude": 127.6750, "roi_group": "lake_basin", "water_context": "lake", "terrain_context": "basin"},
    {"city": "chuncheon_east", "base_city": "chuncheon", "latitude": 37.9100, "longitude": 127.8750, "roi_group": "lake_mountain", "water_context": "lake", "terrain_context": "mountain_edge"},
    {"city": "wonju_west", "base_city": "wonju", "latitude": 37.3350, "longitude": 127.7850, "roi_group": "mountain_inland", "water_context": "river", "terrain_context": "mountain_basin"},
    {"city": "cheongju_west", "base_city": "cheongju", "latitude": 36.6400, "longitude": 127.3500, "roi_group": "inland_plain", "water_context": "river", "terrain_context": "plain"},
    {"city": "cheongju_north", "base_city": "cheongju", "latitude": 36.7550, "longitude": 127.4900, "roi_group": "inland_plain", "water_context": "river", "terrain_context": "plain"},
    {"city": "sejong_west", "base_city": "sejong", "latitude": 36.5000, "longitude": 127.1650, "roi_group": "river_inland", "water_context": "river", "terrain_context": "plain"},
    {"city": "jeonju_west", "base_city": "jeonju", "latitude": 35.8350, "longitude": 127.0200, "roi_group": "inland_plain", "water_context": "river", "terrain_context": "plain"},
    {"city": "andong_lake", "base_city": "andong", "latitude": 36.5850, "longitude": 128.8400, "roi_group": "river_lake", "water_context": "lake", "terrain_context": "basin"},
    {"city": "gangneung_coast", "base_city": "gangneung", "latitude": 37.7750, "longitude": 128.9650, "roi_group": "east_coast", "water_context": "coast", "terrain_context": "coastal_plain"},
    {"city": "gangneung_mountain", "base_city": "gangneung", "latitude": 37.7050, "longitude": 128.7700, "roi_group": "east_coast_mountain", "water_context": "river", "terrain_context": "coastal_mountain"},
    {"city": "pohang_industrial", "base_city": "pohang", "latitude": 35.9800, "longitude": 129.4000, "roi_group": "east_coast_industrial", "water_context": "coast", "terrain_context": "coastal_plain"},
    {"city": "pohang_inland", "base_city": "pohang", "latitude": 36.0800, "longitude": 129.2300, "roi_group": "east_coast_basin", "water_context": "river", "terrain_context": "coastal_basin"},
    {"city": "ulsan_industrial", "base_city": "ulsan", "latitude": 35.4850, "longitude": 129.3650, "roi_group": "coastal_industrial", "water_context": "coast", "terrain_context": "coastal_plain"},
    {"city": "ulsan_inland", "base_city": "ulsan", "latitude": 35.6000, "longitude": 129.1850, "roi_group": "coastal_basin", "water_context": "river", "terrain_context": "basin"},
    {"city": "busan_west", "base_city": "busan", "latitude": 35.1250, "longitude": 128.9700, "roi_group": "coastal_metro", "water_context": "river_coast", "terrain_context": "coastal_plain"},
    {"city": "busan_east", "base_city": "busan", "latitude": 35.2000, "longitude": 129.2050, "roi_group": "coastal_metro", "water_context": "coast", "terrain_context": "coastal_mountain"},
    {"city": "changwon_bay", "base_city": "changwon", "latitude": 35.1450, "longitude": 128.6550, "roi_group": "south_coast_industrial", "water_context": "bay", "terrain_context": "coastal_plain"},
    {"city": "gimhae_plain", "base_city": "gimhae", "latitude": 35.3000, "longitude": 128.8000, "roi_group": "river_coastal", "water_context": "river", "terrain_context": "plain"},
    {"city": "gunsan_coast", "base_city": "gunsan", "latitude": 35.9550, "longitude": 126.5950, "roi_group": "west_coast", "water_context": "coast", "terrain_context": "coastal_plain"},
    {"city": "mokpo_inner_bay", "base_city": "mokpo", "latitude": 34.7850, "longitude": 126.2950, "roi_group": "southwest_coast", "water_context": "bay", "terrain_context": "coastal_plain"},
    {"city": "yeosu_industrial", "base_city": "yeosu", "latitude": 34.8350, "longitude": 127.7050, "roi_group": "south_coast_industrial", "water_context": "coast", "terrain_context": "coastal_hill"},
    {"city": "suncheon_bay", "base_city": "suncheon", "latitude": 34.8750, "longitude": 127.5150, "roi_group": "bay_wetland", "water_context": "bay_wetland", "terrain_context": "coastal_plain"},
    {"city": "ansan_sihwa", "base_city": "ansan", "latitude": 37.3000, "longitude": 126.6800, "roi_group": "west_coast_industrial", "water_context": "lake_coast", "terrain_context": "coastal_plain"},
    {"city": "pyeongtaek_port", "base_city": "pyeongtaek", "latitude": 36.9650, "longitude": 126.8450, "roi_group": "west_coast_industrial", "water_context": "coast", "terrain_context": "coastal_plain"},
    {"city": "cheonan_east", "base_city": "cheonan", "latitude": 36.8150, "longitude": 127.2500, "roi_group": "inland_plain", "water_context": "river", "terrain_context": "plain"},
    {"city": "asan_bay", "base_city": "asan", "latitude": 36.8900, "longitude": 126.9100, "roi_group": "west_coast", "water_context": "bay", "terrain_context": "coastal_plain"},
    {"city": "dangjin_coast", "base_city": "dangjin", "latitude": 36.9850, "longitude": 126.6300, "roi_group": "west_coast_industrial", "water_context": "coast", "terrain_context": "coastal_plain"},
    {"city": "seosan_coast", "base_city": "seosan", "latitude": 36.7550, "longitude": 126.3350, "roi_group": "west_coast", "water_context": "coast", "terrain_context": "coastal_plain"},
    {"city": "gumi_industrial", "base_city": "gumi", "latitude": 36.0950, "longitude": 128.4300, "roi_group": "river_industrial", "water_context": "river", "terrain_context": "basin"},
    {"city": "gyeongju_coast", "base_city": "gyeongju", "latitude": 35.7600, "longitude": 129.4550, "roi_group": "east_coast", "water_context": "coast", "terrain_context": "coastal_basin"},
    {"city": "geoje_north", "base_city": "geoje", "latitude": 34.9550, "longitude": 128.5900, "roi_group": "island_coast", "water_context": "coast", "terrain_context": "coastal_hill"},
    {"city": "tongyeong_bay", "base_city": "tongyeong", "latitude": 34.8300, "longitude": 128.3500, "roi_group": "south_coast", "water_context": "bay", "terrain_context": "coastal_hill"},
    {"city": "sacheon_bay", "base_city": "sacheon", "latitude": 34.9650, "longitude": 127.9950, "roi_group": "south_coast", "water_context": "bay", "terrain_context": "coastal_plain"},
    {"city": "haenam_coast", "base_city": "haenam", "latitude": 34.4850, "longitude": 126.5050, "roi_group": "southwest_coast", "water_context": "coast", "terrain_context": "coastal_plain"},
]

# attempt3 추가 보강 ROI: 1차 extra가 기존 15.36km 패치와 많이 겹쳐서 탈락하므로,
# 기존 도심에서 충분히 떨어진 산업단지/하천/해안/호수/산지 ROI 후보를 더 넣는다.
EXTRA_ROI_CENTERS.extend([
    {"city": "goyang_south", "base_city": "goyang", "latitude": 37.6000, "longitude": 126.8400, "roi_group": "metro_river", "water_context": "river", "terrain_context": "plain"},
    {"city": "goyang_hangang", "base_city": "goyang", "latitude": 37.6500, "longitude": 126.7600, "roi_group": "metro_river", "water_context": "river", "terrain_context": "plain"},
    {"city": "gimpo_hangang", "base_city": "gimpo", "latitude": 37.6200, "longitude": 126.7100, "roi_group": "metro_river", "water_context": "river", "terrain_context": "plain"},
    {"city": "gimpo_coast", "base_city": "gimpo", "latitude": 37.6400, "longitude": 126.5600, "roi_group": "west_coast", "water_context": "coast_river", "terrain_context": "coastal_plain"},
    {"city": "siheung_sihwa", "base_city": "siheung", "latitude": 37.3400, "longitude": 126.7300, "roi_group": "west_coast_industrial", "water_context": "lake_coast", "terrain_context": "coastal_plain"},
    {"city": "hwaseong_sihwa", "base_city": "hwaseong", "latitude": 37.1850, "longitude": 126.6850, "roi_group": "west_coast_industrial", "water_context": "coast_lake", "terrain_context": "coastal_plain"},
    {"city": "hwaseong_inland", "base_city": "hwaseong", "latitude": 37.1600, "longitude": 126.9300, "roi_group": "metro_inland", "water_context": "river", "terrain_context": "plain"},
    {"city": "osn_inland", "base_city": "osan", "latitude": 37.1500, "longitude": 127.0700, "roi_group": "metro_inland", "water_context": "river", "terrain_context": "plain"},
    {"city": "anseong_plain", "base_city": "anseong", "latitude": 37.0100, "longitude": 127.2700, "roi_group": "inland_plain", "water_context": "river", "terrain_context": "plain"},
    {"city": "yangpyeong_river", "base_city": "yangpyeong", "latitude": 37.4900, "longitude": 127.4900, "roi_group": "river_mountain", "water_context": "river", "terrain_context": "mountain_basin"},
    {"city": "gapyeong_lake", "base_city": "gapyeong", "latitude": 37.7300, "longitude": 127.5200, "roi_group": "lake_mountain", "water_context": "lake", "terrain_context": "mountain"},
    {"city": "hongcheon_basin", "base_city": "hongcheon", "latitude": 37.7000, "longitude": 127.8900, "roi_group": "mountain_basin", "water_context": "river", "terrain_context": "mountain_basin"},
    {"city": "hoengseong_basin", "base_city": "hoengseong", "latitude": 37.4900, "longitude": 127.9850, "roi_group": "mountain_basin", "water_context": "river", "terrain_context": "mountain_basin"},
    {"city": "yeoju_river", "base_city": "yeoju", "latitude": 37.3000, "longitude": 127.6400, "roi_group": "river_inland", "water_context": "river", "terrain_context": "plain"},
    {"city": "jincheon_plain", "base_city": "jincheon", "latitude": 36.8550, "longitude": 127.4350, "roi_group": "inland_plain", "water_context": "river", "terrain_context": "plain"},
    {"city": "eumseong_plain", "base_city": "eumseong", "latitude": 36.9400, "longitude": 127.6900, "roi_group": "inland_plain", "water_context": "river", "terrain_context": "plain"},
    {"city": "goesan_mountain", "base_city": "goesan", "latitude": 36.8150, "longitude": 127.7900, "roi_group": "mountain_inland", "water_context": "river", "terrain_context": "mountain"},
    {"city": "boeun_basin", "base_city": "boeun", "latitude": 36.4900, "longitude": 127.7300, "roi_group": "mountain_basin", "water_context": "river", "terrain_context": "basin"},
    {"city": "okcheon_river", "base_city": "okcheon", "latitude": 36.3050, "longitude": 127.5700, "roi_group": "river_basin", "water_context": "river", "terrain_context": "basin"},
    {"city": "geumsan_basin", "base_city": "geumsan", "latitude": 36.1050, "longitude": 127.4900, "roi_group": "mountain_basin", "water_context": "river", "terrain_context": "basin"},
    {"city": "gyeryong_basin", "base_city": "gyeryong", "latitude": 36.2750, "longitude": 127.2450, "roi_group": "basin_city", "water_context": "river", "terrain_context": "mountain_edge"},
    {"city": "gongju_river", "base_city": "gongju", "latitude": 36.4550, "longitude": 127.1250, "roi_group": "river_inland", "water_context": "river", "terrain_context": "basin"},
    {"city": "buyeo_river", "base_city": "buyeo", "latitude": 36.2750, "longitude": 126.9100, "roi_group": "river_plain", "water_context": "river", "terrain_context": "plain"},
    {"city": "nonsan_plain", "base_city": "nonsan", "latitude": 36.1900, "longitude": 127.0950, "roi_group": "inland_plain", "water_context": "river", "terrain_context": "plain"},
    {"city": "yesan_plain", "base_city": "yesan", "latitude": 36.6800, "longitude": 126.8450, "roi_group": "inland_plain", "water_context": "river_lake", "terrain_context": "plain"},
    {"city": "hongseong_coast", "base_city": "hongseong", "latitude": 36.5700, "longitude": 126.6200, "roi_group": "west_coast", "water_context": "coast", "terrain_context": "coastal_plain"},
    {"city": "taean_coast", "base_city": "taean", "latitude": 36.7450, "longitude": 126.2950, "roi_group": "west_coast", "water_context": "coast", "terrain_context": "coastal_plain"},
    {"city": "taean_south", "base_city": "taean", "latitude": 36.5200, "longitude": 126.3300, "roi_group": "west_coast", "water_context": "coast", "terrain_context": "coastal_plain"},
    {"city": "jeongeup_plain", "base_city": "jeongeup", "latitude": 35.5700, "longitude": 126.8550, "roi_group": "inland_plain", "water_context": "river", "terrain_context": "plain"},
    {"city": "gimje_plain", "base_city": "gimje", "latitude": 35.8000, "longitude": 126.8850, "roi_group": "inland_plain", "water_context": "river", "terrain_context": "plain"},
    {"city": "buan_coast", "base_city": "buan", "latitude": 35.7250, "longitude": 126.6200, "roi_group": "west_coast", "water_context": "coast", "terrain_context": "coastal_plain"},
    {"city": "gochang_coast", "base_city": "gochang", "latitude": 35.4350, "longitude": 126.5200, "roi_group": "west_coast", "water_context": "coast", "terrain_context": "coastal_plain"},
    {"city": "sunchang_basin", "base_city": "sunchang", "latitude": 35.3750, "longitude": 127.1400, "roi_group": "mountain_basin", "water_context": "river", "terrain_context": "basin"},
    {"city": "jangseong_plain", "base_city": "jangseong", "latitude": 35.3000, "longitude": 126.7850, "roi_group": "inland_plain", "water_context": "river", "terrain_context": "plain"},
    {"city": "hampyeong_plain", "base_city": "hampyeong", "latitude": 35.0650, "longitude": 126.5200, "roi_group": "river_plain", "water_context": "river", "terrain_context": "plain"},
    {"city": "muan_coast", "base_city": "muan", "latitude": 34.9900, "longitude": 126.3950, "roi_group": "southwest_coast", "water_context": "coast", "terrain_context": "coastal_plain"},
    {"city": "yeongam_plain", "base_city": "yeongam", "latitude": 34.8000, "longitude": 126.6900, "roi_group": "river_plain", "water_context": "river", "terrain_context": "plain"},
    {"city": "gangjin_coast", "base_city": "gangjin", "latitude": 34.6400, "longitude": 126.7650, "roi_group": "southwest_coast", "water_context": "coast", "terrain_context": "coastal_plain"},
    {"city": "jindo_coast", "base_city": "jindo", "latitude": 34.4800, "longitude": 126.2600, "roi_group": "island_coast", "water_context": "coast", "terrain_context": "coastal_hill"},
    {"city": "gurye_basin", "base_city": "gurye", "latitude": 35.2050, "longitude": 127.4650, "roi_group": "mountain_basin", "water_context": "river", "terrain_context": "basin"},
    {"city": "gwangyang_coast", "base_city": "gwangyang", "latitude": 34.9400, "longitude": 127.6950, "roi_group": "south_coast_industrial", "water_context": "coast", "terrain_context": "coastal_plain"},
    {"city": "sunchang_river", "base_city": "sunchang", "latitude": 35.2750, "longitude": 127.2350, "roi_group": "river_basin", "water_context": "river", "terrain_context": "basin"},
    {"city": "hamyang_basin", "base_city": "hamyang", "latitude": 35.5200, "longitude": 127.7250, "roi_group": "mountain_basin", "water_context": "river", "terrain_context": "basin"},
    {"city": "geochang_basin", "base_city": "geochang", "latitude": 35.6850, "longitude": 127.9100, "roi_group": "mountain_basin", "water_context": "river", "terrain_context": "basin"},
    {"city": "hapcheon_basin", "base_city": "hapcheon", "latitude": 35.5650, "longitude": 128.1650, "roi_group": "river_basin", "water_context": "river_lake", "terrain_context": "basin"},
    {"city": "uiseong_basin", "base_city": "uiseong", "latitude": 36.3550, "longitude": 128.6950, "roi_group": "river_basin", "water_context": "river", "terrain_context": "basin"},
    {"city": "cheongsong_mountain", "base_city": "cheongsong", "latitude": 36.4350, "longitude": 129.0600, "roi_group": "mountain_inland", "water_context": "river", "terrain_context": "mountain"},
    {"city": "yeongdeok_coast", "base_city": "yeongdeok", "latitude": 36.4150, "longitude": 129.3650, "roi_group": "east_coast", "water_context": "coast", "terrain_context": "coastal_mountain"},
    {"city": "uljin_coast", "base_city": "uljin", "latitude": 36.9950, "longitude": 129.4000, "roi_group": "east_coast", "water_context": "coast", "terrain_context": "coastal_mountain"},
    {"city": "yeongju_basin", "base_city": "yeongju", "latitude": 36.8050, "longitude": 128.6250, "roi_group": "mountain_basin", "water_context": "river", "terrain_context": "basin"},
    {"city": "mungyeong_basin", "base_city": "mungyeong", "latitude": 36.5950, "longitude": 128.1950, "roi_group": "mountain_basin", "water_context": "river", "terrain_context": "basin"},
    {"city": "yongdeok_mountain", "base_city": "yeongdeok", "latitude": 36.3400, "longitude": 129.1300, "roi_group": "east_coast_mountain", "water_context": "river", "terrain_context": "mountain"},
    {"city": "yangyang_coast", "base_city": "yangyang", "latitude": 38.0150, "longitude": 128.6900, "roi_group": "east_coast", "water_context": "coast", "terrain_context": "coastal_mountain"},
    {"city": "inje_mountain", "base_city": "inje", "latitude": 38.0000, "longitude": 128.1700, "roi_group": "mountain_inland", "water_context": "river", "terrain_context": "mountain"},
    {"city": "jeongseon_mountain", "base_city": "jeongseon", "latitude": 37.3800, "longitude": 128.6600, "roi_group": "mountain_inland", "water_context": "river", "terrain_context": "mountain"},
    {"city": "goseong_south_coast", "base_city": "goseong_gn", "latitude": 34.9750, "longitude": 128.3200, "roi_group": "south_coast", "water_context": "coast", "terrain_context": "coastal_hill"},
    {"city": "namhae_coast", "base_city": "namhae", "latitude": 34.8350, "longitude": 127.8950, "roi_group": "island_coast", "water_context": "coast", "terrain_context": "coastal_hill"},
    {"city": "wando_coast", "base_city": "wando", "latitude": 34.3150, "longitude": 126.7550, "roi_group": "island_coast", "water_context": "coast", "terrain_context": "coastal_hill"},
])


def ensure_base_city(rows):
    out = []
    for row in rows:
        row = dict(row)
        row.setdefault("base_city", row["city"])
        out.append(row)
    return out


def roi_projected_bbox(row):
    to_xy = Transformer.from_crs("EPSG:4326", CRS_BASE, always_xy=True)
    cx, cy = to_xy.transform(float(row["longitude"]), float(row["latitude"]))
    half = GRID_N * PIXEL_SIZE_M / 2.0
    return {
        "left": cx - half,
        "right": cx + half,
        "bottom": cy - half,
        "top": cy + half,
    }


def projected_bboxes_overlap(a, b, gap_m=MIN_ROI_GAP_M):
    return not (
        a["right"] + gap_m <= b["left"]
        or b["right"] + gap_m <= a["left"]
        or a["top"] + gap_m <= b["bottom"]
        or b["top"] + gap_m <= a["bottom"]
    )


def build_nonoverlap_roi_centers(base_rows, extra_rows, target_count=TARGET_ROI_COUNT):
    selected = ensure_base_city(base_rows)
    rejected = []
    seen = {row["city"] for row in selected}
    selected_boxes = [(row["city"], roi_projected_bbox(row)) for row in selected]
    for row in ensure_base_city(extra_rows):
        if len(selected) >= target_count:
            break
        if row["city"] in seen:
            rejected.append({**row, "reject_reason": "duplicate_city"})
            continue
        box = roi_projected_bbox(row)
        overlaps = [name for name, prev_box in selected_boxes if projected_bboxes_overlap(box, prev_box)]
        if overlaps:
            rejected.append({**row, "reject_reason": "patch_bbox_overlap:" + ";".join(overlaps[:5])})
            continue
        selected.append(row)
        selected_boxes.append((row["city"], box))
        seen.add(row["city"])
    report = pd.DataFrame(rejected)
    if not report.empty:
        report.to_csv(PROJECT_DIR / "attempt3" / "roi_overlap_rejected.csv", index=False, encoding="utf-8-sig")
    print("ROI centers base/extra/selected/rejected:", len(base_rows), len(extra_rows), len(selected), len(rejected))
    if len(selected) < target_count:
        print(f"WARNING: selected ROI count {len(selected)} is below target {target_count}. Add more non-overlapping EXTRA_ROI_CENTERS if needed.")
    return selected

CITY_CENTERS = build_nonoverlap_roi_centers(CITY_CENTERS, EXTRA_ROI_CENTERS)

# EGIS 중분류/세분류 WMS가 민통선 북부에서 공백으로 들어오는 ROI는 후보에서 제외한다.
# 기존 고양-의정부-동두천-포천 직선 기준은 너무 남쪽이라 정상 ROI까지 잘릴 수 있어 쓰지 않는다.
DMZ_EAST_WEST_SPLIT = {
    # 38도 07분 30초 N, 127도 16분 00초 E
    # 이 경도 동쪽은 ROI상세.png의 기존 북방한계를 유지하고, 서쪽은 더 남쪽 보수 기준을 쓴다.
    "name": "east_west_split_38_07_30N_127_16_00E",
    "longitude": 127.2666667,
    "latitude": 38.1250000,
}
DMZ_WEST_LOWER_LIMIT_POINTS = [
    {"name": "west_grid_start_lower", "longitude": 126.82, "latitude": 37.62},
    {"name": "uijeongbu_lower", "longitude": 127.03, "latitude": 37.70},
    {"name": "pocheon_lower", "longitude": 127.20, "latitude": 37.86},
    DMZ_EAST_WEST_SPLIT,
]
DMZ_EAST_NORTHERN_LIMIT_POINTS = [
    DMZ_EAST_WEST_SPLIT,
    {"name": "cheorwon_hwacheon_top_grid", "longitude": 127.65, "latitude": 38.12},
    {"name": "yanggu_inje_top_grid", "longitude": 128.15, "latitude": 38.12},
    {"name": "inje_yangyang_top_grid", "longitude": 128.55, "latitude": 38.12},
    # 동해안은 정암-낙산 중간보다 약간 북쪽, 속초 남쪽 격자 상단에 맞춘다.
    {"name": "sokcho_yangyang_top_grid", "longitude": 128.75, "latitude": 38.12},
    {"name": "east_coast_top_grid", "longitude": 129.15, "latitude": 38.12},
]
# ROI bbox top이 이 margin 이상 북방한계를 넘으면 제외한다. 작은 수치 오차/투영 오차는 허용한다.
DMZ_NORTHERN_LIMIT_MARGIN_DEG = 0.005
ROI_NORTHERN_LIMIT_DETAIL_CSV = PROJECT_DIR / "attempt3" / "roi_northern_limit_detail.csv"


print("ROI centers defined for attempt3:", len(CITY_CENTERS), "target:", TARGET_ROI_COUNT)
print("OUT_CSV:", OUT_CSV)


ROI centers base/extra/selected/rejected: 56 104 100 60
ROI centers defined for attempt3: 100 target: 100
OUT_CSV: c:\Users\shinh\eigenspace\Workspace\26.05_MLproj\attempt3\lst_candidates.csv


## 2. Earth Engine 초기화와 ROI

In [18]:
EE_PROJECT = os.environ.get("EE_PROJECT", "neat-acre-496003-i5")
try:
    ee.Initialize(project=EE_PROJECT)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=EE_PROJECT)

def validate_kma_auth_key(auth_key):
    auth_key = str(auth_key).strip()
    # 좌표/문장/공백이 들어가면 KMA가 401을 내기 전에 즉시 중단한다.
    bad_tokens = ["°"]
    if not auth_key or any(tok in auth_key for tok in bad_tokens):
        raise ValueError(
            "KMA APIHub authKey looks invalid. "
            "좌표나 설명문이 아니라 KMA APIHub 인증키 문자열을 입력해야 합니다."
        )
    if len(auth_key) < 10:
        raise ValueError("KMA APIHub authKey is too short")
    return auth_key

KMA_AUTH_KEY = os.environ.get("KMA_APIHUB_AUTH_KEY", "").strip()
if not KMA_AUTH_KEY:
    KMA_AUTH_KEY = getpass("KMA APIHub authKey: ").strip()
KMA_AUTH_KEY = validate_kma_auth_key(KMA_AUTH_KEY)


def patch_projected_bounds(center_lat, center_lon, grid_n=GRID_N, pixel_size_m=PIXEL_SIZE_M, crs_base=CRS_BASE):
    to_xy = Transformer.from_crs("EPSG:4326", crs_base, always_xy=True)
    cx, cy = to_xy.transform(center_lon, center_lat)
    half = grid_n * pixel_size_m / 2
    return cx - half, cy - half, cx + half, cy + half


def patch_lonlat_bbox(center_lat, center_lon, grid_n=GRID_N, pixel_size_m=PIXEL_SIZE_M, crs_base=CRS_BASE):
    left, bottom, right, top = patch_projected_bounds(center_lat, center_lon, grid_n, pixel_size_m, crs_base)
    to_ll = Transformer.from_crs(crs_base, "EPSG:4326", always_xy=True)
    corners = [to_ll.transform(x, y) for x, y in [(left, bottom), (left, top), (right, bottom), (right, top)]]
    lons = [pt[0] for pt in corners]
    lats = [pt[1] for pt in corners]
    return {"min_lon": min(lons), "min_lat": min(lats), "max_lon": max(lons), "max_lat": max(lats)}


def bbox_intersects(a, b):
    return not (
        a["max_lon"] < b["min_lon"] or a["min_lon"] > b["max_lon"]
        or a["max_lat"] < b["min_lat"] or a["min_lat"] > b["max_lat"]
    )


def interpolate_cutoff_lat(points, lon):
    pts = sorted(points, key=lambda p: p["longitude"])
    if lon <= pts[0]["longitude"]:
        return float(pts[0]["latitude"]), pts[0]["name"]
    if lon >= pts[-1]["longitude"]:
        return float(pts[-1]["latitude"]), pts[-1]["name"]
    for left, right in zip(pts[:-1], pts[1:]):
        if left["longitude"] <= lon <= right["longitude"]:
            denom = right["longitude"] - left["longitude"]
            if abs(denom) <= 1e-12:
                return max(float(left["latitude"]), float(right["latitude"])), f"{left['name']}->{right['name']}"
            t = (lon - left["longitude"]) / denom
            lat = float(left["latitude"] + t * (right["latitude"] - left["latitude"]))
            return lat, f"{left['name']}->{right['name']}"
    return float(pts[-1]["latitude"]), pts[-1]["name"]


def dmz_cutoff_lat_for_lon(lon):
    lon = float(lon)
    if lon < float(DMZ_EAST_WEST_SPLIT["longitude"]):
        lat, segment = interpolate_cutoff_lat(DMZ_WEST_LOWER_LIMIT_POINTS, lon)
        return lat, "west_lower:" + segment
    lat, segment = interpolate_cutoff_lat(DMZ_EAST_NORTHERN_LIMIT_POINTS, lon)
    return lat, "east_original:" + segment


def dmz_exclusion_detail(center_lat, center_lon):
    roi_bbox = patch_lonlat_bbox(center_lat, center_lon)
    # ROI가 북방한계와 겹치면 제외해야 하므로 center lon이 아니라 ROI 좌/중/우 lon을 모두 본다.
    probe_lons = [roi_bbox["min_lon"], (roi_bbox["min_lon"] + roi_bbox["max_lon"]) / 2, roi_bbox["max_lon"]]
    probe_rows = []
    hits = []
    for lon in probe_lons:
        cutoff_lat, rule_name = dmz_cutoff_lat_for_lon(lon)
        margin = roi_bbox["max_lat"] - cutoff_lat
        hit = margin >= DMZ_NORTHERN_LIMIT_MARGIN_DEG
        probe_rows.append({
            "probe_lon": float(lon),
            "cutoff_lat": float(cutoff_lat),
            "cutoff_segment": rule_name,
            "roi_top_minus_cutoff_deg": float(margin),
            "hit": bool(hit),
        })
        if hit:
            hits.append(f"{rule_name}@{cutoff_lat:.4f}/margin={margin:.4f}")
    hits = sorted(set(hits))
    return hits, roi_bbox, probe_rows


def dmz_exclusion_hit(center_lat, center_lon):
    hits, roi_bbox, _ = dmz_exclusion_detail(center_lat, center_lon)
    return hits, roi_bbox


def filter_supported_city_centers(city_centers):
    kept, excluded, detail_rows = [], [], []
    for city in city_centers:
        hits, roi_bbox, probe_rows = dmz_exclusion_detail(city["latitude"], city["longitude"])
        city = dict(city)
        city.update({
            "roi_min_lon": roi_bbox["min_lon"],
            "roi_min_lat": roi_bbox["min_lat"],
            "roi_max_lon": roi_bbox["max_lon"],
            "roi_max_lat": roi_bbox["max_lat"],
            "dmz_exclusion_hit": ";".join(hits),
        })
        max_margin = max(row["roi_top_minus_cutoff_deg"] for row in probe_rows)
        city["dmz_max_margin_deg"] = float(max_margin)
        for row in probe_rows:
            detail_rows.append({
                "city": city["city"],
                "latitude": city["latitude"],
                "longitude": city["longitude"],
                "roi_min_lon": roi_bbox["min_lon"],
                "roi_min_lat": roi_bbox["min_lat"],
                "roi_max_lon": roi_bbox["max_lon"],
                "roi_max_lat": roi_bbox["max_lat"],
                **row,
            })
        if hits:
            excluded.append(city)
        else:
            kept.append(city)

    detail_df = pd.DataFrame(detail_rows).sort_values(["roi_top_minus_cutoff_deg", "city"], ascending=[False, True])
    ROI_NORTHERN_LIMIT_DETAIL_CSV.parent.mkdir(parents=True, exist_ok=True)
    detail_df.to_csv(ROI_NORTHERN_LIMIT_DETAIL_CSV, index=False, encoding="utf-8-sig")
    print("ROI centers kept/excluded by image-based northern limit:", len(kept), len(excluded))
    print("Northern limit detail:", ROI_NORTHERN_LIMIT_DETAIL_CSV)
    display_cols = [
        "city", "latitude", "longitude", "roi_min_lat", "roi_max_lat",
        "dmz_max_margin_deg", "dmz_exclusion_hit",
    ]
    if excluded:
        display(pd.DataFrame(excluded)[display_cols])
    # 북방한계에 가까운 ROI를 항상 보여줘서 이미지 기준이 맞는지 바로 확인한다.
    display(detail_df.head(20))
    return kept, excluded


ACTIVE_CITY_CENTERS, EXCLUDED_CITY_CENTERS = filter_supported_city_centers(CITY_CENTERS)


def make_patch_roi(center_lat, center_lon, grid_n=GRID_N, pixel_size_m=PIXEL_SIZE_M, crs_base=CRS_BASE):
    left, bottom, right, top = patch_projected_bounds(center_lat, center_lon, grid_n, pixel_size_m, crs_base)
    return ee.Geometry.Rectangle([left, bottom, right, top], proj=crs_base, geodesic=False)


def cloud_mask_landsat(image):
    qa = image.select("QA_PIXEL")
    bad_bits = [0, 1, 2, 3, 4]  # fill, dilated cloud, cirrus, cloud, cloud shadow
    mask = ee.Image(1)
    for bit in bad_bits:
        mask = mask.And(qa.bitwiseAnd(1 << bit).eq(0))
    return mask.rename("clear_mask")


ROI centers kept/excluded by image-based northern limit: 98 2
Northern limit detail: c:\Users\shinh\eigenspace\Workspace\26.05_MLproj\attempt3\roi_northern_limit_detail.csv


,city,latitude,longitude,roi_min_lat,roi_max_lat,dmz_max_margin_deg,dmz_exclusion_hit
0,goyang_hangang,37.65,126.76,37.575416,37.72455,0.10455,west_lower:west_grid_start_lower->uijeongbu_lo...
1,gimpo_coast,37.64,126.56,37.565307,37.71466,0.09466,west_lower:west_grid_start_lower@37.6200/margi...


,city,latitude,longitude,roi_min_lon,roi_min_lat,roi_max_lon,roi_max_lat,probe_lon,cutoff_lat,cutoff_segment,roi_top_minus_cutoff_deg,hit
177,goyang_hangang,37.6500,126.7600,126.666140,37.575416,126.853705,37.724550,126.666140,37.620000,west_lower:west_grid_start_lower,0.104550,True
178,goyang_hangang,37.6500,126.7600,126.666140,37.575416,126.853705,37.724550,126.759923,37.620000,west_lower:west_grid_start_lower,0.104550,True
180,gimpo_coast,37.6400,126.5600,126.466015,37.565307,126.653831,37.714660,126.466015,37.620000,west_lower:west_grid_start_lower,0.094660,True
181,gimpo_coast,37.6400,126.5600,126.466015,37.565307,126.653831,37.714660,126.559923,37.620000,west_lower:west_grid_start_lower,0.094660,True
182,gimpo_coast,37.6400,126.5600,126.466015,37.565307,126.653831,37.714660,126.653831,37.620000,west_lower:west_grid_start_lower,0.094660,True
179,goyang_hangang,37.6500,126.7600,126.666140,37.575416,126.853705,37.724550,126.853705,37.632840,west_lower:west_grid_start_lower->uijeongbu_lower,0.091710,True
0,seoul,37.5665,126.9780,126.884412,37.492048,127.071434,37.640917,126.884412,37.644538,west_lower:west_grid_start_lower->uijeongbu_lower,-0.003620,False
285,yangyang_coast,38.0150,128.6900,128.597037,37.941495,128.782803,38.088463,128.597037,38.120000,east_original:inje_yangyang_top_grid->sokcho_y...,-0.031537,False
286,yangyang_coast,38.0150,128.6900,128.597037,37.941495,128.782803,38.088463,128.689920,38.120000,east_original:inje_yangyang_top_grid->sokcho_y...,-0.031537,False
287,yangyang_coast,38.0150,128.6900,128.597037,37.941495,128.782803,38.088463,128.782803,38.120000,east_original:sokcho_yangyang_top_grid->east_c...,-0.031537,False


## 3. AWS 기후자료 밀도 기준


In [19]:
AWSH_BASE_URL = "https://apihub.kma.go.kr/api/typ01/url/awsh.php"
STN_BASE_URL = "https://apihub.kma.go.kr/api/typ01/url/stn_inf.php"
AWS_GROUPS = {
    "temp": ["TA"],
    "wind": ["WD", "WS"],
    "rain_1h": ["RN"],
    "humidity": ["HM"],
}
AWSH_COLUMNS = {"TA": ["TM", "STN", "TA"], "WD": ["TM", "STN", "WD"], "WS": ["TM", "STN", "WS"], "RN": ["TM", "STN", "RN"], "HM": ["TM", "STN", "HM"]}
_aws_cache = {}


def decode_kma_response(content):
    for enc in ["utf-8", "cp949", "euc-kr"]:
        try:
            return content.decode(enc)
        except UnicodeDecodeError:
            continue
    return content.decode("utf-8", errors="replace")


def valid_data_lines(text):
    return [ln.strip() for ln in text.splitlines() if ln.strip() and not ln.startswith("#")]


def fetch_awsh_var(var, tm, auth_key):
    params = {"var": var, "tm": tm, "stn": "0", "help": "0", "authKey": auth_key}
    r = requests.get(AWSH_BASE_URL, params=params, timeout=60)
    r.raise_for_status()
    lines = valid_data_lines(decode_kma_response(r.content))
    rows = [re.split(r"\s+", ln)[: len(AWSH_COLUMNS[var])] for ln in lines]
    df = pd.DataFrame(rows, columns=AWSH_COLUMNS[var]) if rows else pd.DataFrame(columns=AWSH_COLUMNS[var])
    for c in df.columns:
        if c != "TM":
            df[c] = pd.to_numeric(df[c], errors="coerce")
    return df


def fetch_station_info(auth_key, tm):
    params = {"inf": "AWS", "stn": "", "tm": tm, "help": "1", "authKey": auth_key}
    r = requests.get(STN_BASE_URL, params=params, timeout=60)
    r.raise_for_status()
    lines = valid_data_lines(decode_kma_response(r.content))
    rows = []
    for ln in lines:
        parts = re.split(r"\s+", ln)
        nums = [x for x in parts if re.fullmatch(r"-?\d+(\.\d+)?", x)]
        if len(nums) >= 3:
            rows.append({"STN": int(float(nums[0])), "LON": float(nums[1]), "LAT": float(nums[2])})
    return pd.DataFrame(rows).drop_duplicates("STN")


def aws_frame_for_date(date_ymd):
    tm = pd.Timestamp(date_ymd).strftime("%Y%m%d") + HOUR_KST
    if tm in _aws_cache:
        return _aws_cache[tm]
    aws = fetch_station_info(KMA_AUTH_KEY, tm)
    for var in AWSH_COLUMNS:
        df = fetch_awsh_var(var, tm, KMA_AUTH_KEY)
        aws = aws.merge(df[["STN", var]], on="STN", how="left")
    _aws_cache[tm] = aws.dropna(subset=["LON", "LAT"])
    return _aws_cache[tm]


def aws_density_metrics(date_ymd, center_lat, center_lon):
    aws = aws_frame_for_date(date_ymd).copy()
    if aws.empty:
        return {"aws_pass": False, "aws_nearest_km": np.inf, **{f"aws_{g}_count_50km": 0 for g in AWS_GROUPS}}
    to_xy = Transformer.from_crs("EPSG:4326", CRS_BASE, always_xy=True)
    cx, cy = to_xy.transform(center_lon, center_lat)
    sx, sy = to_xy.transform(aws["LON"].to_numpy(), aws["LAT"].to_numpy())
    dist_km = np.sqrt((sx - cx) ** 2 + (sy - cy) ** 2) / 1000.0
    aws["dist_km"] = dist_km
    metrics = {"aws_nearest_km": float(np.nanmin(dist_km))}
    pass_groups = []
    for group, cols in AWS_GROUPS.items():
        usable = aws.loc[(aws["dist_km"] <= AWS_DENSITY_RADIUS_KM) & aws[cols].notna().all(axis=1)]
        count = int(len(usable))
        metrics[f"aws_{group}_count_50km"] = count
        pass_groups.append(count >= MIN_AWS_STATIONS_PER_GROUP)
    metrics["aws_pass"] = bool(all(pass_groups) and metrics["aws_nearest_km"] <= MAX_NEAREST_AWS_DISTANCE_KM)
    return metrics


## 3. Landsat 장면 검색

In [20]:
def merged_landsat_collection(roi):
    collections = []
    for col_id in LANDSAT_COLLECTIONS:
        collections.append(
            ee.ImageCollection(col_id)
            .filterBounds(roi)
            .filterDate(START_DATE, END_DATE)
            .filter(ee.Filter.lt("CLOUD_COVER", MAX_SCENE_CLOUD_COVER))
        )
    col = collections[0]
    for part in collections[1:]:
        col = col.merge(part)
    return col


def scene_clear_fraction(image, roi):
    clear = cloud_mask_landsat(image)
    stats = clear.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=roi,
        scale=60,
        maxPixels=1_000_000,
        bestEffort=True,
    )
    return image.set("roi_clear_fraction", stats.get("clear_mask"))


def scene_row(image_info, city_info):
    props = image_info["properties"]
    product_id = props.get("LANDSAT_PRODUCT_ID") or image_info["id"].split("/")[-1]
    ts = pd.to_datetime(props["system:time_start"], unit="ms", utc=True).tz_convert("Asia/Seoul")
    return {
        "city": city_info["city"],
        "base_city": city_info.get("base_city", city_info["city"]),
        "latitude": city_info["latitude"],
        "longitude": city_info["longitude"],
        "roi_group": city_info.get("roi_group", "unspecified"),
        "water_context": city_info.get("water_context", "unspecified"),
        "terrain_context": city_info.get("terrain_context", "unspecified"),
        "roi_min_lon": city_info.get("roi_min_lon"),
        "roi_min_lat": city_info.get("roi_min_lat"),
        "roi_max_lon": city_info.get("roi_max_lon"),
        "roi_max_lat": city_info.get("roi_max_lat"),
        "date": ts.strftime("%Y-%m-%d"),
        "year": int(ts.year),
        "landsat_product_id": product_id,
        "scene_id": image_info["id"],
        "collection": "/".join(image_info["id"].split("/")[:-1]),
        "cloud_cover_scene": props.get("CLOUD_COVER"),
        "clear_fraction_roi": props.get("roi_clear_fraction"),
        "sun_azimuth": props.get("SUN_AZIMUTH"),
        "sun_elevation": props.get("SUN_ELEVATION"),
    }


def collect_city_candidates(city_info):
    roi = make_patch_roi(city_info["latitude"], city_info["longitude"])
    col = merged_landsat_collection(roi).map(lambda img: scene_clear_fraction(img, roi))
    col = col.filter(ee.Filter.gte("roi_clear_fraction", MIN_CLEAR_FRACTION))
    info = col.sort("system:time_start").getInfo()
    rows = [scene_row(feature, city_info) for feature in info["features"]]
    return pd.DataFrame(rows)


## 4. 후보 수집과 연도별 제한

In [21]:
SEASON_MONTHS = {
    "winter": [12, 1, 2],
    "spring": [3, 4, 5],
    "summer": [6, 7, 8],
    "autumn": [9, 10, 11],
}
SEASON_ORDER = {"winter": 0, "spring": 1, "summer": 2, "autumn": 3}


def season_name(month):
    return "spring" if month in [3, 4, 5] else "summer" if month in [6, 7, 8] else "autumn" if month in [9, 10, 11] else "winter"


def season_day_bounds(year, season):
    year = int(year)
    if season == "winter":
        start = pd.Timestamp(year=year, month=1, day=1)
        end = pd.Timestamp(year=year, month=2, day=28) + pd.offsets.MonthEnd(0)
    else:
        months = SEASON_MONTHS[season]
        start = pd.Timestamp(year=year, month=months[0], day=1)
        end = pd.Timestamp(year=year, month=months[-1], day=1) + pd.offsets.MonthEnd(0)
    return start, end


def stable_uniform01(*parts):
    key = "|".join(map(str, parts)).encode("utf-8")
    digest = hashlib.blake2b(key, digest_size=8).digest()
    value = int.from_bytes(digest, "big")
    return value / float(2 ** 64 - 1)


def target_date_for_city_year_season(city, year, season, city_order, n_cities):
    start, end = season_day_bounds(year, season)
    span_days = max((end - start).days, 1)
    # ROI-year-season별 고정 난수로 계절 내부 날짜를 균등하게 뿌린다.
    # 재실행해도 같은 seed에서는 같은 날짜 target이 나오므로 후보표 재현성이 유지된다.
    u = stable_uniform01(ROI_BALANCE_SEED, city, int(year), season)
    offset_days = int(round(span_days * u))
    return start + pd.Timedelta(days=offset_days)


def select_one_city_year_season(part, used_date_counts, city_order, n_cities):
    part = part.copy()
    city = str(part["city"].iloc[0])
    year = int(part["year"].iloc[0])
    season = str(part["season"].iloc[0])
    target = target_date_for_city_year_season(city, year, season, city_order, n_cities)
    part["date_ts"] = pd.to_datetime(part["date"])
    part["target_date"] = target
    part["date_distance_days"] = (part["date_ts"] - target).abs().dt.days.astype(float)
    part["date_used_count_before"] = part["date"].map(lambda d: used_date_counts.get(str(d), 0)).astype(float)
    clear_rank = 1.0 - part["clear_fraction_roi"].astype(float).clip(0, 1)
    aws_rank = part["aws_nearest_km"].astype(float).fillna(999.0) / 100.0
    # 날짜 중복 회피가 최우선, 그 다음 계절 내 목표 날짜 근접, 그 다음 clear/AWS 품질.
    part["balance_score"] = (
        DATE_OVERLAP_PENALTY * part["date_used_count_before"]
        + part["date_distance_days"]
        + clear_rank
        + aws_rank
    )
    chosen = part.sort_values(["balance_score", "date_distance_days", "clear_fraction_roi", "aws_nearest_km"], ascending=[True, True, False, True]).iloc[0].copy()
    used_date_counts[str(chosen["date"])] = used_date_counts.get(str(chosen["date"]), 0) + 1
    return chosen


def pre_aws_shortlist(df_city, city_info, city_order, n_cities):
    """EE cloud/날짜 기준으로 ROI-year-season별 AWS 검사 대표 1개만 남긴다."""
    if df_city.empty:
        return df_city.copy()
    df = df_city.copy()
    df["month"] = pd.to_datetime(df["date"]).dt.month
    df["season"] = df["month"].map(season_name)
    df["date_ts"] = pd.to_datetime(df["date"])
    targets = []
    for row in df.itertuples(index=False):
        targets.append(target_date_for_city_year_season(row.city, int(row.year), row.season, city_order, n_cities))
    df["target_date"] = targets
    df["date_distance_days"] = (df["date_ts"] - df["target_date"]).abs().dt.days.astype(float)
    # AWS 호출 전에는 AWS 정보가 없으므로 clear_fraction과 계절 내 목표 날짜 근접성만으로 대표를 고른다.
    df = df.sort_values(
        ["city", "year", "season", "date_distance_days", "clear_fraction_roi", "date"],
        ascending=[True, True, True, True, False, True],
    )
    df = df.groupby(["city", "year", "season"], as_index=False, group_keys=False).head(PRE_AWS_SCENES_PER_ROI_YEAR_SEASON)
    duplicated = df.duplicated(["city", "year", "season"]).sum()
    if duplicated:
        raise RuntimeError(f"pre-AWS representative selection failed: {duplicated} duplicated ROI-year-season rows")
    return df.reset_index(drop=True)


all_rows = []
roi_pass_counts = []
active_city_names = [str(city["city"]) for city in ACTIVE_CITY_CENTERS]
city_order = {city: idx for idx, city in enumerate(active_city_names)}
n_cities = len(active_city_names)
expected_max_samples = n_cities * (pd.Timestamp(END_DATE).year - pd.Timestamp(START_DATE).year + 1) * 4 * SAMPLES_PER_CITY_YEAR_SEASON
print("target ROI count:", TARGET_ROI_COUNT, "active ROI count:", n_cities)
print("target max samples:", expected_max_samples, "= ROI x years x seasons x per_group")
print("AWS density calls are capped to one fixed 2016-01-01 reference check per ROI")

for city in tqdm(ACTIVE_CITY_CENTERS):
    df_city = collect_city_candidates(city)
    raw_count = len(df_city)
    if df_city.empty:
        print("no Landsat candidates:", city["city"])
        roi_pass_counts.append({"city": city["city"], "base_city": city.get("base_city", city["city"]), "raw_candidates": raw_count, "pre_aws_candidates": 0, "aws_pass_candidates": 0, "aws_density_reference_date": AWS_DENSITY_REFERENCE_DATE})
        continue

    df_city = pre_aws_shortlist(df_city, city, city_order, n_cities)
    pre_aws_count = len(df_city)  # ROI-year-season EE representatives; not AWS call count
    if df_city.empty:
        print("no pre-AWS shortlist candidates:", city["city"])
        roi_pass_counts.append({"city": city["city"], "base_city": city.get("base_city", city["city"]), "raw_candidates": raw_count, "pre_aws_candidates": 0, "aws_pass_candidates": 0, "aws_density_reference_date": AWS_DENSITY_REFERENCE_DATE})
        continue

    density = aws_density_metrics(AWS_DENSITY_REFERENCE_DATE, city["latitude"], city["longitude"])
    density_record = {"aws_density_reference_date": AWS_DENSITY_REFERENCE_DATE, **density}
    if not density["aws_pass"]:
        print("no AWS-dense ROI:", city["city"], density_record)
        roi_pass_counts.append({"city": city["city"], "base_city": city.get("base_city", city["city"]), "raw_candidates": raw_count, "pre_aws_candidates": pre_aws_count, "aws_pass_candidates": 0, **density_record})
        continue

    for key, value in density_record.items():
        df_city[key] = value
    df_city = df_city.sort_values(["date", "clear_fraction_roi", "aws_nearest_km"], ascending=[True, False, True])
    df_city = df_city.drop_duplicates(["city", "date"], keep="first").reset_index(drop=True)
    roi_pass_counts.append({"city": city["city"], "base_city": city.get("base_city", city["city"]), "raw_candidates": raw_count, "pre_aws_candidates": pre_aws_count, "aws_pass_candidates": len(df_city), **density_record})
    all_rows.append(df_city)

candidates_all = pd.concat(all_rows, ignore_index=True) if all_rows else pd.DataFrame()
roi_pass_df = pd.DataFrame(roi_pass_counts)
print("ROI pass counts")
display(roi_pass_df.sort_values("aws_pass_candidates"))

if candidates_all.empty:
    candidates = pd.DataFrame()
else:
    group_counts = candidates_all.groupby(["city", "year", "season"]).size().reset_index(name="candidate_count")
    print("city-year-season groups with candidates:", len(group_counts))
    print("expected max groups:", expected_max_samples)
    display(group_counts.groupby("city")["candidate_count"].agg(["count", "sum", "min", "median", "max"]).sort_values("count"))

    used_date_counts = {}
    picked_rows = []
    # 후보가 적은 그룹부터 고르면 희소 ROI/계절이 먼저 좋은 날짜를 확보한다.
    for _, group_row in group_counts.sort_values(["candidate_count", "city", "year", "season"]).iterrows():
        part = candidates_all[
            (candidates_all["city"] == group_row["city"])
            & (candidates_all["year"] == group_row["year"])
            & (candidates_all["season"] == group_row["season"])
        ]
        for _ in range(int(SAMPLES_PER_CITY_YEAR_SEASON)):
            if part.empty:
                break
            chosen = select_one_city_year_season(part, used_date_counts, city_order, n_cities)
            picked_rows.append(chosen)
            part = part.drop(index=chosen.name, errors="ignore")
    candidates = pd.DataFrame(picked_rows).reset_index(drop=True)
    candidates["date_used_count_final"] = candidates["date"].map(candidates["date"].value_counts()).astype(int)
    candidates["roi_candidate_count_before_balance"] = candidates["city"].map(roi_pass_df.set_index("city")["aws_pass_candidates"])

if not candidates.empty:
    before = len(candidates)
    candidates = (
        candidates
        .sort_values(["city", "year", "season", "date", "clear_fraction_roi", "aws_nearest_km"], ascending=[True, True, True, True, False, True])
        .drop_duplicates(["city", "year", "season"], keep="first")
        .reset_index(drop=True)
    )
    print("deduplicated city-year-season scenes:", before, "->", len(candidates))
print(candidates.shape)
print(candidates.groupby("city").size().describe() if not candidates.empty else "no candidates")
print("date overlap summary")
print(candidates["date"].value_counts().describe() if not candidates.empty else "no candidates")
print(candidates[["roi_group", "water_context", "terrain_context"]].value_counts().head(30) if not candidates.empty else "no candidates")
candidates.head()


target ROI count: 100 active ROI count: 98
target max samples: 3920 = ROI x years x seasons x per_group
AWS density calls are capped to one fixed 2016-01-01 reference check per ROI


  0%|          | 0/98 [00:00<?, ?it/s]

ROI pass counts


,city,base_city,raw_candidates,pre_aws_candidates,aws_pass_candidates,aws_density_reference_date,aws_nearest_km,aws_temp_count_50km,aws_wind_count_50km,aws_rain_1h_count_50km,aws_humidity_count_50km,aws_pass
0,seoul,seoul,183,39,39,2016-01-01,1.215252,122,122,122,46,True
70,nonsan_plain,nonsan,351,40,40,2016-01-01,2.692274,32,32,32,26,True
69,buyeo_river,buyeo,362,40,40,2016-01-01,1.017062,30,30,30,23,True
68,geumsan_basin,geumsan,171,40,40,2016-01-01,0.750060,32,32,32,22,True
67,okcheon_river,okcheon,173,40,40,2016-01-01,2.462319,34,34,34,25,True
...,...,...,...,...,...,...,...,...,...,...,...,...
28,yongin,yongin,354,40,40,2016-01-01,1.503385,112,112,112,45,True
27,suncheon,suncheon,170,40,40,2016-01-01,2.194902,34,34,34,26,True
26,yeosu,yeosu,175,40,40,2016-01-01,7.593161,24,24,24,16,True
24,gunsan,gunsan,355,40,40,2016-01-01,4.762008,29,29,29,20,True


city-year-season groups with candidates: 3919
expected max groups: 3920


,count,sum,min,median,max
city,,,,,
seoul,39,39,1,1.0,1
andong,40,40,1,1.0,1
pyeongtaek,40,40,1,1.0,1
pyeongchang,40,40,1,1.0,1
pohang,40,40,1,1.0,1
...,...,...,...,...,...
gimcheon,40,40,1,1.0,1
geumsan_basin,40,40,1,1.0,1
geoje,40,40,1,1.0,1


deduplicated city-year-season scenes: 3919 -> 3919
(3919, 36)
count    98.000000
mean     39.989796
std       0.101015
min      39.000000
25%      40.000000
50%      40.000000
75%      40.000000
max      40.000000
dtype: float64
date overlap summary
count    643.000000
mean       6.094868
std        5.444025
min        1.000000
25%        2.000000
50%        5.000000
75%        8.000000
max       38.000000
Name: count, dtype: float64
roi_group               water_context  terrain_context 
inland_plain            river          plain               480
mountain_basin          river          basin               320
west_coast              coast          coastal_plain       240
east_coast              coast          coastal_mountain    240
river_basin             river          basin               240
mountain_inland         river          mountain            200
west_coast_industrial   coast          coastal_plain       160
river_plain             river          plain               160
is

,city,base_city,latitude,longitude,roi_group,water_context,terrain_context,roi_min_lon,roi_min_lat,roi_max_lon,...,aws_nearest_km,aws_temp_count_50km,aws_wind_count_50km,aws_rain_1h_count_50km,aws_humidity_count_50km,aws_pass,date_used_count_before,balance_score,date_used_count_final,roi_candidate_count_before_balance
0,andong,andong,36.5684,128.7294,river_basin,river_lake,basin,128.638422,36.495054,128.820229,...,2.045565,22,22,22,13,True,0.0,6.023358,19,40
1,andong,andong,36.5684,128.7294,river_basin,river_lake,basin,128.638422,36.495054,128.820229,...,2.045565,22,22,22,13,True,0.0,19.038278,5,40
2,andong,andong,36.5684,128.7294,river_basin,river_lake,basin,128.638422,36.495054,128.820229,...,2.045565,22,22,22,13,True,0.0,0.406161,4,40
3,andong,andong,36.5684,128.7294,river_basin,river_lake,basin,128.638422,36.495054,128.820229,...,2.045565,22,22,22,13,True,0.0,2.021899,38,40
4,andong,andong,36.5684,128.7294,river_basin,river_lake,basin,128.638422,36.495054,128.820229,...,2.045565,22,22,22,13,True,0.0,1.317185,12,40


## 5. split 부여

In [22]:
def assign_split_by_date(df, val_frac=0.15, test_frac=0.15):
    df = df.sort_values(["date", "city"]).reset_index(drop=True)
    unique_dates = np.array(sorted(df["date"].unique()))
    rng = np.random.default_rng(42)
    shuffled = unique_dates.copy()
    rng.shuffle(shuffled)
    n = len(shuffled)
    n_test = max(1, int(round(n * test_frac))) if n else 0
    n_val = max(1, int(round(n * val_frac))) if n else 0
    test_dates = set(shuffled[:n_test])
    val_dates = set(shuffled[n_test:n_test + n_val])
    df["split"] = "train"
    df.loc[df["date"].isin(val_dates), "split"] = "val"
    df.loc[df["date"].isin(test_dates), "split"] = "test"
    return df

if candidates.empty:
    raise RuntimeError("No candidates collected. Lower MIN_CLEAR_FRACTION or check CITY_CENTERS/date range.")

candidates = assign_split_by_date(candidates)
candidates = candidates.sort_values(["split", "city", "date"]).reset_index(drop=True)
print(candidates["split"].value_counts())
candidates[["city", "base_city", "roi_group", "water_context", "terrain_context", "split", "date", "landsat_product_id", "clear_fraction_roi", "aws_nearest_km", "aws_temp_count_50km", "aws_wind_count_50km", "aws_rain_1h_count_50km", "aws_humidity_count_50km"]].head(20)


split
train    2834
val       603
test      482
Name: count, dtype: int64


,city,base_city,roi_group,water_context,terrain_context,split,date,landsat_product_id,clear_fraction_roi,aws_nearest_km,aws_temp_count_50km,aws_wind_count_50km,aws_rain_1h_count_50km,aws_humidity_count_50km
0,andong,andong,river_basin,river_lake,basin,test,2016-03-02,LC08_L2SP_114035_20160302_20200907_02_T1,0.982178,2.045565,22,22,22,13
1,andong,andong,river_basin,river_lake,basin,test,2016-08-09,LC08_L2SP_114035_20160809_20200906_02_T1,0.614295,2.045565,22,22,22,13
2,andong,andong,river_basin,river_lake,basin,test,2017-06-25,LC08_L2SP_114035_20170625_20200903_02_T1,0.326546,2.045565,22,22,22,13
3,andong,andong,river_basin,river_lake,basin,test,2018-02-04,LC08_L2SP_114035_20180204_20200902_02_T1,0.957283,2.045565,22,22,22,13
4,andong,andong,river_basin,river_lake,basin,test,2018-10-02,LC08_L2SP_114035_20181002_20200830_02_T1,0.876902,2.045565,22,22,22,13
5,andong,andong,river_basin,river_lake,basin,test,2022-07-25,LC08_L2SP_114035_20220725_20220802_02_T1,0.630006,2.045565,22,22,22,13
6,andong,andong,river_basin,river_lake,basin,test,2023-03-30,LC09_L2SP_114035_20230330_20230401_02_T1,0.360944,2.045565,22,22,22,13
7,andong,andong,river_basin,river_lake,basin,test,2024-01-04,LC08_L2SP_114035_20240104_20240114_02_T1,0.967103,2.045565,22,22,22,13
8,andong,andong,river_basin,river_lake,basin,test,2024-06-04,LC09_L2SP_114035_20240604_20240605_02_T1,0.919946,2.045565,22,22,22,13
9,andong,andong,river_basin,river_lake,basin,test,2025-09-27,LC09_L2SP_114035_20250927_20250928_02_T1,0.908651,2.045565,22,22,22,13


## 6. 저장

In [23]:
OUT_CSV.parent.mkdir(parents=True, exist_ok=True)
candidates.to_csv(OUT_CSV, index=False, encoding="utf-8-sig")
try:
    candidates.to_excel(OUT_XLSX, index=False)
except Exception as exc:
    print("xlsx save skipped:", exc)
print("saved:", OUT_CSV)
print("saved:", OUT_XLSX if OUT_XLSX.exists() else "xlsx not written")


saved: c:\Users\shinh\eigenspace\Workspace\26.05_MLproj\attempt3\lst_candidates.csv
saved: c:\Users\shinh\eigenspace\Workspace\26.05_MLproj\attempt3\lst_candidates.xlsx
